#### OpenAQ Tokyo PM2.5 to CSV

Source: https://explore.openaq.org/login

This notebook shows how to use the **OpenAQ v3 API** to collect **PM2.5** data for the **Tokyo area** and save it as CSV. It first finds PM2.5 locations/sensors near Tokyo, then downloads hourly PM2.5 data for a chosen time window.

**Important Note: Data Collection Issues & API Rate Limits**

While running this notebook, the data collection process for Step 2 (Hourly PM2.5 Data) was partially unsuccessful due to API rate limiting.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/test/

Mounted at /content/drive
/content/drive/MyDrive/test


In [ ]:
# !pip install requests pandas
import time
import requests
import pandas as pd
from pathlib import Path

API_KEY = 'Please obtain your own API key and enter it here.' # Please obtain your own API key and enter it here.
TOKYO_LAT = 35.6762
TOKYO_LON = 139.6503
RADIUS_METERS = 25000
DATETIME_FROM = '2025-01-01T00:00:00Z'
DATETIME_TO = '2025-01-31T23:59:59Z'
LIMIT_PER_PAGE = 1000
OUTPUT_DIR = Path('output')
OUTPUT_DIR.mkdir(exist_ok=True)
BASE_URL = 'https://api.openaq.org/v3'
HEADERS = {'X-API-Key': API_KEY}

if API_KEY == 'PASTE_YOUR_OPENAQ_API_KEY_HERE':
    raise ValueError('Please paste your OpenAQ API key before running the notebook.')

def fetch_all_pages(endpoint: str, params: dict, headers: dict, limit_per_page: int = 1000, sleep_seconds: float = 0.2):
    all_results = []
    page = 1
    while True:
        query = dict(params)
        query['page'] = page
        query['limit'] = limit_per_page
        response = requests.get(endpoint, headers=headers, params=query, timeout=60)
        response.raise_for_status()
        payload = response.json()
        results = payload.get('results', [])
        if not results:
            break
        all_results.extend(results)
        print(f'Fetched page {page}: {len(results)} rows')
        if len(results) < limit_per_page:
            break
        page += 1
        time.sleep(sleep_seconds)
    return all_results

## Step 1: Find Tokyo-area PM2.5 locations

This query uses the **locations** endpoint with `coordinates`, `radius`, and `parameters_id=2` (PM2.5). The code flattens the response and saves `tokyo_pm25_locations.csv`.

In [ ]:
locations_raw = fetch_all_pages(
    f'{BASE_URL}/locations',
    {'coordinates': f'{TOKYO_LAT},{TOKYO_LON}', 'radius': RADIUS_METERS, 'parameters_id': 2, 'order_by': 'id', 'sort_order': 'asc'},
    HEADERS,
    LIMIT_PER_PAGE
)

location_rows = []
for loc in locations_raw:
    for sensor in (loc.get('sensors') or []):
        if (sensor.get('parameter') or {}).get('name') == 'pm25':
            location_rows.append({
                'location_id': loc.get('id'),
                'location_name': loc.get('name'),
                'locality': loc.get('locality'),
                'timezone': loc.get('timezone'),
                'country_code': (loc.get('country') or {}).get('code'),
                'country_name': (loc.get('country') or {}).get('name'),
                'provider_name': (loc.get('provider') or {}).get('name'),
                'latitude': (loc.get('coordinates') or {}).get('latitude'),
                'longitude': (loc.get('coordinates') or {}).get('longitude'),
                'sensor_id': sensor.get('id'),
                'sensor_name': sensor.get('name'),
                'parameter_name': (sensor.get('parameter') or {}).get('name'),
                'parameter_display_name': (sensor.get('parameter') or {}).get('displayName'),
                'units': (sensor.get('parameter') or {}).get('units')
            })

locations_df = pd.DataFrame(location_rows).drop_duplicates()
locations_df = locations_df.sort_values(['location_id', 'sensor_id']).reset_index(drop=True) if not locations_df.empty else locations_df
locations_csv = OUTPUT_DIR / 'tokyo_pm25_locations.csv'
locations_df.to_csv(locations_csv, index=False, encoding='utf-8-sig')
print('PM2.5 sensors found:', len(locations_df))
print('Saved:', locations_csv.resolve())
locations_df.head()

Fetched page 1: 127 rows
PM2.5 sensors found: 127
Saved: /content/drive/MyDrive/test/output/tokyo_pm25_locations.csv


,location_id,location_name,locality,timezone,country_code,country_name,provider_name,latitude,longitude,sensor_id,sensor_name,parameter_name,parameter_display_name,units
0,1214487,三鷹市下連雀７丁目１５番４号,,Asia/Tokyo,JP,Japan,Ministry of the Environment Air Pollutant Wide...,35.688889,139.563889,6518561,pm25 µg/m³,pm25,PM2.5,µg/m³
1,1214507,世田谷区世田谷４－２１－２７,,Asia/Tokyo,JP,Japan,Ministry of the Environment Air Pollutant Wide...,35.644444,139.655556,6516560,pm25 µg/m³,pm25,PM2.5,µg/m³
2,1214509,世田谷区粕谷２－１９,,Asia/Tokyo,JP,Japan,Ministry of the Environment Air Pollutant Wide...,35.661111,139.616667,6520423,pm25 µg/m³,pm25,PM2.5,µg/m³
3,1214510,中央区新川一丁目３番１号,,Asia/Tokyo,JP,Japan,Ministry of the Environment Air Pollutant Wide...,35.675000,139.786111,6516386,pm25 µg/m³,pm25,PM2.5,µg/m³
4,1214511,中央区晴海三丁目６番１号,,Asia/Tokyo,JP,Japan,Ministry of the Environment Air Pollutant Wide...,35.652778,139.783333,6520768,pm25 µg/m³,pm25,PM2.5,µg/m³


## Step 2: Download hourly PM2.5 data and save CSV

For each PM2.5 sensor, this code calls the **hourly** endpoint (`/v3/sensors/{sensor_id}/hours`) with `datetime_from` and `datetime_to`, then saves `tokyo_pm25_hourly.csv`.

In [ ]:
def fetch_hourly_pm25_for_sensor(sensor_id: int, datetime_from: str, datetime_to: str):
    return fetch_all_pages(
        f'{BASE_URL}/sensors/{sensor_id}/hours',
        {'datetime_from': datetime_from, 'datetime_to': datetime_to},
        HEADERS,
        LIMIT_PER_PAGE
    )

hourly_rows = []
unique_sensor_ids = locations_df['sensor_id'].dropna().astype(int).unique().tolist() if not locations_df.empty else []
for i, sensor_id in enumerate(unique_sensor_ids, start=1):
    print(f'[{i}/{len(unique_sensor_ids)}] sensor {sensor_id}')
    try:
        measurements = fetch_hourly_pm25_for_sensor(sensor_id, DATETIME_FROM, DATETIME_TO)
        meta = locations_df.loc[locations_df['sensor_id'] == sensor_id].iloc[0].to_dict()
        for m in measurements:
            hourly_rows.append({
                'location_id': meta.get('location_id'),
                'location_name': meta.get('location_name'),
                'sensor_id': sensor_id,
                'sensor_name': meta.get('sensor_name'),
                'parameter_name': meta.get('parameter_name', 'pm25'),
                'units': meta.get('units'),
                'datetime_utc': ((m.get('datetime') or {}).get('utc')),
                'datetime_local': ((m.get('datetime') or {}).get('local')),
                'value': m.get('value'),
                'latitude': (m.get('coordinates') or {}).get('latitude'),
                'longitude': (m.get('coordinates') or {}).get('longitude')
            })
    except Exception as e:
        print(f'Error for sensor {sensor_id}: {e}')

hourly_df = pd.DataFrame(hourly_rows)
hourly_df = hourly_df.sort_values(['sensor_id', 'datetime_utc']).reset_index(drop=True) if not hourly_df.empty else hourly_df
hourly_csv = OUTPUT_DIR / 'tokyo_pm25_hourly.csv'
hourly_df.to_csv(hourly_csv, index=False, encoding='utf-8-sig')
print('Hourly rows:', len(hourly_df))
print('Saved:', hourly_csv.resolve())
print('Unique locations:', hourly_df['location_id'].nunique() if not hourly_df.empty else 0)
print('Unique sensors:', hourly_df['sensor_id'].nunique() if not hourly_df.empty else 0)
hourly_df.head()

[1/127] sensor 6518561
Fetched page 1: 668 rows
[2/127] sensor 6516560
[3/127] sensor 6520423
Fetched page 1: 670 rows
[4/127] sensor 6516386
Fetched page 1: 632 rows
[5/127] sensor 6520768
Fetched page 1: 668 rows
[6/127] sensor 6520559
Fetched page 1: 671 rows
[7/127] sensor 6518459
Fetched page 1: 672 rows
[8/127] sensor 6520247
Fetched page 1: 677 rows
[9/127] sensor 6518123
Fetched page 1: 672 rows
[10/127] sensor 6515093
Fetched page 1: 678 rows
[11/127] sensor 6520155
Fetched page 1: 669 rows
[12/127] sensor 6515094
Fetched page 1: 631 rows
[13/127] sensor 6520834
Fetched page 1: 679 rows
[14/127] sensor 6520476
Fetched page 1: 678 rows
[15/127] sensor 6515670
Fetched page 1: 677 rows
[16/127] sensor 6517887
Fetched page 1: 581 rows
[17/127] sensor 6516135
Fetched page 1: 631 rows
[18/127] sensor 6515657
Fetched page 1: 663 rows
[19/127] sensor 6518218
Fetched page 1: 671 rows
[20/127] sensor 6517783
Fetched page 1: 668 rows
[21/127] sensor 6520414
Fetched page 1: 588 rows
[22/1

,location_id,location_name,sensor_id,sensor_name,parameter_name,units,datetime_utc,datetime_local,value,latitude,longitude
0,1215460,横浜市都筑区茅ヶ崎中央３２－１,6515020,pm25 µg/m³,pm25,µg/m³,None,None,4.0,None,None
1,1215460,横浜市都筑区茅ヶ崎中央３２－１,6515020,pm25 µg/m³,pm25,µg/m³,None,None,2.0,None,None
2,1215460,横浜市都筑区茅ヶ崎中央３２－１,6515020,pm25 µg/m³,pm25,µg/m³,None,None,6.0,None,None
3,1215460,横浜市都筑区茅ヶ崎中央３２－１,6515020,pm25 µg/m³,pm25,µg/m³,None,None,9.0,None,None
4,1215460,横浜市都筑区茅ヶ崎中央３２－１,6515020,pm25 µg/m³,pm25,µg/m³,None,None,3.0,None,None
